# Chapter 24 — Branching and Forking

**Companion to *Pi Agents*** · [`Pi Agents` chapter 24](https://programmer.ie/books/pi/24-chapter/)

| | |
|---|---|
| Chapter | `24-chapter.md` · weight 24 · `/books/pi/24-chapter/` |
| Notebook | `notebooks/pi/24-chapter.ipynb` |
| Pin | `@earendil-works/pi-*` **1.0.4** (`examples/package.json`) |
| Ground truth | `docs/*.md` and `dist/*.d.ts` **at the pin**, under `examples/node_modules/` |

> **Where the authority lives.** Every claim below is checked against the
> packages installed under `examples/node_modules/@earendil-works/`, not against
> a globally installed Pi and not from memory. When prose and an exported
> declaration disagree, the declaration wins.

## The question

After leaving a branch, what is still in the tree, what stops reaching the
model, and what does a fork write?

The chapter's point: `/tree` moves within the session file, `/fork` creates a new
session from an earlier user message, `/clone` copies the active branch into a
new session. The tree stores everything; the active branch is what reaches the
model. A branch you did not choose changes what the model sees.


## What this notebook establishes

- The session is an **append-only tree of entries with stable ids**. Branching creates new children from an earlier entry; the current position is the leaf.
- `/tree` on a **user message** returns its text to the editor and moves the leaf to the message's parent (new sibling branch). On an **assistant response**, it continues after that entry with an empty editor.
- Submitting an edited prompt creates a **sibling**: two message entries under the same parent, in one session file.
- The **abandoned branch stays** in the tree — it is not deleted. A `branch_summary` entry can be appended to the new branch, recording the `fromId` of the abandoned leaf.
- `/fork` creates a **new session file** from an earlier user message — the abandoned path does NOT carry over. The response includes the original prompt text for the editor.
- `/clone` copies the **active branch only** into a new session file — abandoned branches do not come with it.
- `get_entries` returns **all entries** (including pre-compaction history and abandoned branches); `get_messages` returns only the active branch.
- Branch-relative state: `ContextEditEntry` edits are branch-relative — navigating to before the edit reveals the original. Extensions must rebuild state from `getBranch()`, not every file entry.
- Labels are bookmarks on entries and appear in `get_tree`.

## What this notebook does **not** establish

- **Branch summary quality not evaluated** — the test checks structure, not content.
- **The `branchSummary.skipPrompt` setting** is documented, not run.
- **`session_before_fork` extension cancellation** is documented, not run.
- **RPC `fork`/`clone` over shipped binary** is tested but the extension cancellation path is not exercised.


## Setup

This notebook runs the book's own examples. **Pi is Node/TypeScript**; Python
only presents, orchestrates and checks. There is no Python re-implementation
of Pi's loop, context loader, session projection, permissions or protocol
anywhere in this set — where a mechanism is demonstrated, the code that runs is
the code in `examples/`.

* **Execution mode:** **Run** — real sessions with the shared persisted history, branching in-process and via RPC against the shipped binary.
* **Evidence scope:** `in_process_runtime` + `shipped_binary`. The tree mechanics and navigation are Pi's.
* **Credentials:** none. The model is a scripted stand-in (`faux`) supplied by
  the book's harness, so nothing here reaches the network, spends money, or
  says anything about how a real model behaves.


In [1]:
# --- Locate the shared support module and the pinned packages -----------------
# Two documented markers, both overridable:
#   PIN_REPO       -> the checkout of the pi repository
#   PIN_NOTEBOOKS  -> this notebook directory
# Because both searches walk upwards from the working directory, this cell
# works whether you launched Jupyter from notebooks/pi/ or from the repo root.
import sys
from pathlib import Path


def _find_support() -> Path:
    for c in (Path.cwd(), *Path.cwd().parents):
        if (c / "pinb.py").is_file():
            return c
        nested = c / "notebooks" / "pi"
        if (nested / "pinb.py").is_file():
            return nested
    raise SystemExit(
        "Could not find notebooks/pi/pinb.py from " + str(Path.cwd()) + ".\n"
        "Open this notebook from notebooks/pi/, or set PIN_NOTEBOOKS."
    )


sys.path.insert(0, str(_find_support()))
import pinb  # noqa: E402  (a local module, deliberately after the path fix)

pinb.self_check()


{'repo_root': 'resolved',
 'notebook_dir': 'resolved',
 'pin_matches_manuscript': True,
 'node': 'v25.8.2',
 'python': '3.13.15',
 'claim_ledger': True,
 'credentials_required': False}

In [2]:
pinb.show_environment("This notebook's runtime")

print()
print("Repository marker: an ancestor directory holding both")
print("  content/books/pi/                          (the manuscript)")
print("  examples/node_modules/@earendil-works/pi-coding-agent   (the pin)")
print("Every path below is shown relative to that repository; no absolute")
print("path is printed, so the saved output is portable.")
print()
print(pinb.md_table(
    ["resource", "repository-relative path"],
    [["manuscript", pinb.rel(pinb.chapter_file(24))],
     ["examples tree", "examples/"],
     ["claim ledger", "examples/evidence.json"],
     ["pinned docs", "examples/node_modules/@earendil-works/pi-coding-agent/docs/"],
     ["pinned declarations", "examples/node_modules/@earendil-works/*/dist/"],
     ["support module", "notebooks/pi/pinb.py"]],
))


== This notebook's runtime ==
component  value
-------------------------------  ----------------------------------------------------
manuscript pin (AGENTS.md)  1.0.4
@earendil-works/pi-ai  1.0.4
@earendil-works/pi-agent-core  1.0.4
@earendil-works/pi-coding-agent  1.0.4
node  v25.8.2
python  3.13.15
platform  win32
examples tree  installed
model used  faux (scripted) - no network, no credential, no cost

Repository marker: an ancestor directory holding both
  content/books/pi/                          (the manuscript)
  examples/node_modules/@earendil-works/pi-coding-agent   (the pin)
Every path below is shown relative to that repository; no absolute
path is printed, so the saved output is portable.

| resource | repository-relative path |
|---|---|
| manuscript | content/books/pi/24-chapter.md |
| examples tree | examples/ |
| claim ledger | examples/evidence.json |
| pinned docs | examples/node_modules/@earendil-works/pi-coding-agent/docs/ |
| pinned declarations | examples/node_mod

## Baseline: the shared history

In [3]:
BRANCH = pinb.driver_source("ch24-branching.ts")
branch = pinb.driver(BRANCH, label="ch24-branching", timeout=300)
print(pinb.md_table(
    ["operation", "result"],
    [["tree on user message: leaf moved to parent", branch["treeUser"]["leafMovedToParent"]],
     ["tree on user message: sibling created after prompt", branch["siblingCreated"]["siblingsCount"] == 2],
     ["tree on assistant: empty editor", branch["treeAssistant"]["editorEmpty"]],
     ["fork from user message: new session", branch["fork"]["newSessionCreated"]],
     ["fork response has prompt text", branch["fork"]["hasPromptText"]],
     ["clone: new session", branch["clone"]["newSessionCreated"]],
     ["clone copies active branch only", branch["clone"]["activeBranchOnly"]]],
))

| operation | result |
|---|---|
| tree on user message: leaf moved to parent | True |
| tree on user message: sibling created after prompt | True |
| tree on assistant: empty editor | True |
| fork from user message: new session | True |
| fork response has prompt text | True |
| clone: new session | True |
| clone copies active branch only | True |


In [4]:
pinb.show_checks([
    pinb.check("tree on user message: leaf moved to parent",
               branch["treeUser"]["leafMovedToParent"],
               "leaf moved to parent"),
    pinb.check("tree on user message: sibling created after prompt",
               branch["siblingCreated"]["siblingsCount"] == 2,
               "sibling under same parent"),
    pinb.check("tree on assistant: continues after with empty editor",
               branch["treeAssistant"]["editorEmpty"], "empty editor"),
    pinb.check("fork creates new session file", branch["fork"]["newSessionCreated"], "new session"),
    pinb.check("fork response includes original prompt text", branch["fork"]["hasPromptText"], "text for editor"),
    pinb.check("clone creates new session", branch["clone"]["newSessionCreated"], "new session"),
    pinb.check("clone copies active branch only (abandoned branches not included)", branch["clone"]["activeBranchOnly"], "active only"),
])

== Assertions ==
  assertion  observed
----  -----------------------------------------------------------------  -------------------------
PASS  tree on user message: leaf moved to parent  leaf moved to parent
PASS  tree on user message: sibling created after prompt  sibling under same parent
PASS  tree on assistant: continues after with empty editor  empty editor
PASS  fork creates new session file  new session
PASS  fork response includes original prompt text  text for editor
PASS  clone creates new session  new session
PASS  clone copies active branch only (abandoned branches not included)  active only

7/7 assertions held.


## get_entries vs get_messages

In [5]:
print(pinb.md_table(
    ["call", "includes abandoned branches?"],
    [["get_entries", "yes" if branch["entries"]["includesAbandoned"] else "no"],
     ["get_messages", "no" if not branch["messages"]["includesAbandoned"] else "yes"]],
))

| call | includes abandoned branches? |
|---|---|
| get_entries | yes |
| get_messages | no |


In [6]:
pinb.show_checks([
    pinb.check("get_entries includes abandoned branches", branch["entries"]["includesAbandoned"], "all entries"),
    pinb.check("get_messages is active branch only", not branch["messages"]["includesAbandoned"], "active only"),
    pinb.check("get_entries supports since cursor", branch["entries"]["sinceWorks"], "durable cursor"),
])

== Assertions ==
  assertion  observed
----  ---------------------------------------  --------------
PASS  get_entries includes abandoned branches  all entries
PASS  get_messages is active branch only  active only
PASS  get_entries supports since cursor  durable cursor

3/3 assertions held.


## Branch-relative context_edit

In [7]:
print(pinb.md_table(
    ["check", "result"],
    [["edit on branch 1 visible on branch 1", "yes" if branch["edits"]["visibleOnBranch1"] else "no"],
     ["edit on branch 1 NOT visible on branch 2", "no" if not branch["edits"]["visibleOnBranch2"] else "yes"],
     ["navigate to before edit restores original", "yes" if branch["edits"]["restoredOnNavigate"] else "no"]],
))

| check | result |
|---|---|
| edit on branch 1 visible on branch 1 | yes |
| edit on branch 1 NOT visible on branch 2 | no |
| navigate to before edit restores original | yes |


In [8]:
pinb.show_checks([
    pinb.check("context_edit is branch-relative", branch["edits"]["visibleOnBranch1"] and not branch["edits"]["visibleOnBranch2"], "branch-relative"),
    pinb.check("navigating before edit restores original", branch["edits"]["restoredOnNavigate"], "original restored"),
])

== Assertions ==
  assertion  observed
----  ----------------------------------------  -----------------
PASS  context_edit is branch-relative  branch-relative
PASS  navigating before edit restores original  original restored

2/2 assertions held.


## Labels

In [9]:
print(pinb.md_table(
    ["check", "result"],
    [["get_tree reports labels", "yes" if branch["labels"]["inTree"] else "no"],
     ["labelTimestamp present", "yes" if branch["labels"]["hasTimestamp"] else "no"]],
))

| check | result |
|---|---|
| get_tree reports labels | yes |
| labelTimestamp present | yes |


In [10]:
pinb.show_checks([
    pinb.check("labels appear in get_tree", branch["labels"]["inTree"], "labeled"),
    pinb.check("labelTimestamp recorded", branch["labels"]["hasTimestamp"], "timestamped"),
])

== Assertions ==
  assertion  observed
----  -------------------------  -----------
PASS  labels appear in get_tree  labeled
PASS  labelTimestamp recorded  timestamped

2/2 assertions held.


## The chapter's own tests

All tests from the chapter's test file.

## The chapter's canonical evidence

These are the chapter's own tests, executed unmodified through Node's test
runner. Every ledger row the chapter quotes comes from a named case in this
file, so running it here is running the chapter's evidence, not a copy of it.

In [11]:
PATTERNS = 'ch24-branching/branching.test.ts'
tap = pinb.run_tests(PATTERNS, timeout=300)
pinb.show_tap(tap, files='ch24-branching/branching.test.ts', show="branch")

$ node --test ch24-branching/branching.test.ts
  PASS  10 passed, 0 failed, 0 skipped  in 2328 ms

  tests matching 'branch':
    ok   the abandoned branch is not sent, and is not gone
    ok   leaving a branch can attach a summary of it: a branch_summary entry with parentId where you land and fromId where you left
    ok   the summary reaches the model on the new branch, and the abandoned messages still do not
    ok   the tree is real data: getTree() shows both branches under their shared parent
    ok   RPC: clone copies the active branch into a new session


## Your turn: predict, then run

**Predict first.** What happens if you `/fork` from a point that is already on an
abandoned branch? Does the new session include the branch you were on, or the
original active branch?

**Then change one input.** The `branchSummary.reserveTokens` default is 16384.
What happens if you set it to 0 — does the summary still generate?

**Predict the boundary.** `get_entries` with a `since` cursor that matches no
entry id returns `success: false`. What does the client do with that — crash,
retry from start, or show an error?

In [12]:
print("Predictions:")
print("  1. fork from abandoned branch: new session from that point, original active branch not included.")
print("  2. reserveTokens=0: summary still generated (budget applies to input, not output).")
print("  3. invalid since: success: false — client should handle gracefully, not crash.")
print()
print("Observed above:")
print(f"  tree user: leafMoved={branch["treeUser"]["leafMovedToParent"]}")
print(f"  tree assistant: empty editor={branch["treeAssistant"]["editorEmpty"]}")
print(f"  fork: new session={branch["fork"]["newSessionCreated"]}, prompt text={branch["fork"]["hasPromptText"]}")
print(f"  clone: new session={branch["clone"]["newSessionCreated"]}, active only={branch["clone"]["activeBranchOnly"]}")
print(f"  entries: abandoned={branch["entries"]["includesAbandoned"]}")
print(f"  messages: abandoned={branch["messages"]["includesAbandoned"]}")
print(f"  edits: branch-relative={branch["edits"]["visibleOnBranch1"] and not branch["edits"]["visibleOnBranch2"]}")
print()
assert branch["treeUser"]["leafMovedToParent"]
assert branch["fork"]["newSessionCreated"]
assert branch["fork"]["hasPromptText"]
assert branch["clone"]["activeBranchOnly"]
assert branch["entries"]["includesAbandoned"] and not branch["messages"]["includesAbandoned"]
print("held: tree=sibling in same file, fork=new session from user msg, clone=active branch copy; entries=all, messages=active")

Predictions:
  1. fork from abandoned branch: new session from that point, original active branch not included.
  2. reserveTokens=0: summary still generated (budget applies to input, not output).
  3. invalid since: success: false — client should handle gracefully, not crash.

Observed above:
  tree user: leafMoved=True
  tree assistant: empty editor=True
  fork: new session=True, prompt text=True
  clone: new session=True, active only=True
  entries: abandoned=True
  messages: abandoned=False
  edits: branch-relative=True

held: tree=sibling in same file, fork=new session from user msg, clone=active branch copy; entries=all, messages=active


## Interpretation

Branching is three operations at three scales, and the chapter shows the
boundaries:

| Operation | Scale | What it does | The boundary |
|---|---|---|---|
| `/tree` | **Entry** | New sibling under same parent in same file | User msg → edit original; assistant → empty editor after |
| `/fork` | **Session** | New session file from a user message | **Original prompt text returned** for editor; no summary from abandoned path |
| `/clone` | **Session** | New session file from current leaf | **Active branch only** — abandoned branches dropped |

| Read API | What it returns | Use for |
|---|---|---|
| `get_messages` | Active branch only | Normal session display |
| `get_entries` | **All entries** (including abandoned) | Durable cursor, full history |
| `get_tree` | Tree shape with labels | Navigation UI |

State is branch-relative:
* `context_edit` — navigating before edit restores original
* Extension state must be rebuilt from `getBranch()` on `session_start`
* Labels are bookmarks on entries, survive branch changes

Practical rules:
1. **`/tree` for related alternatives** — they stay together, can be summarized forward.
2. **`/fork` for separate work** — new session, no summary carried over.
3. **`/clone` for checkpointing current state** — copy active branch, abandon the rest.
4. **Use `get_entries` for session viewers** — `get_messages` misses abandoned branches.
5. **Rebuild extension state from `getBranch()`** — not from every file entry.

## Sources, execution mode and limitations

**Execution mode:** **Run** — real sessions with the shared persisted history, branching in-process and via RPC against the shipped binary.

**Evidence scope:** `in_process_runtime` + `shipped_binary`. The tree mechanics and navigation are Pi's.

### What was read or run

- **Ran** `drivers/ch24-branching.ts`: real sessions branching the shared history in-process, plus `fork`/`clone` over RPC against shipped binary.
- **Ran** `ch24-branching/branching.test.ts` (10 tests).
- **Read** `examples/evidence.json`, chapter 24 rows (10 claims).

### Limitations

- **Branch summary quality not evaluated** — the test checks structure, not content.
- **The `branchSummary.skipPrompt` setting** is documented, not run.
- **`session_before_fork` extension cancellation** is documented, not run.

Every technical statement in this notebook is about Pi **1.0.4**. A later
release can change any of it; the book's drift protocol in `AGENTS.md` is the
procedure for finding out whether it did.
